In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
ds = xr.open_dataset(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_forcing.nc"))
ds_grid = xr.open_dataset(
    os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_bathymetry_v2.nc")
)

In [ ]:
np_z_centers = (ds_grid.z_faces.values[:-1] + ds_grid.z_faces.values[1:]) / 2
oy, ox, oz = (value for value in ds_grid.sizes.values())
np_mask = (np_z_centers > ds_grid.h.values[..., np.newaxis]).astype(int).transpose(2, 0, 1)

In [ ]:
np_mask_u = np.zeros((oz - 1, oy, ox + 1))
np_mask_u[:, :, :-1] = np_mask
np_mask_u[:, :, 1:] = np.where(np_mask_u[:, :, 1:] == 0, np_mask, np_mask_u[:, :, 1:])

In [ ]:
np_mask_v = np.zeros((oz - 1, oy + 1, ox))
np_mask_v[:, :-1, :] = np_mask
np_mask_v[:, 1:, :] = np.where(np_mask_v[:, 1:, :] == 0, np_mask, np_mask_v[:, 1:, :])

In [ ]:
new_time = pd.date_range(start="2024-01-01T12:00:00", end="2024-12-31T12:00:00", freq="D")

In [ ]:
new_time[~new_time.isin(ds.time.values)]

In [ ]:
ds_new = ds.reindex(time=new_time, method="ffill")

In [ ]:
ds_new

No forcing for temperature and salinity

In [ ]:
ds_new["T_lambda"].values[:] = 0
# ds_new["T_lambda"][:, -2:, :, :] = (1 / (60 * 60 * 24)) * np_mask[-2:, :, :]
ds_new["T_lambda"] = ds_new["T_lambda"].astype(np.float32)

In [ ]:
ds_new["S_lambda"].values[:] = 0
# ds_new["S_lambda"][:, -2:, :100, :] = (1 / (60 * 60 * 24)) * np_mask[-2:, :100, :]
ds_new["S_lambda"] = ds_new["S_lambda"].astype(np.float32)

The southern boundary forcing for u and v

In [ ]:
ds_new["u_lambda"].values[:] = 0
ds_new["u_lambda"][:, :, :10, :] = (1 / (60 * 60 * 24)) * np_mask_u[:, :10, :]
ds_new["u_lambda"] = ds_new["u_lambda"].astype(np.float32)

In [ ]:
ds_new["v_lambda"].values[:] = 0
ds_new["v_lambda"][:, :, :10, :] = (1 / (60 * 60 * 24)) * np_mask_v[:, :10, :]
ds_new["v_lambda"] = ds_new["v_lambda"].astype(np.float32)

Add a river in the Drammenfjord

In [ ]:
ds_new["S_lambda"].values[:, -2:, 186, 8] = 1 / (60 * 60)
ds_new["S"].values[:, -2:, 186, 8] = 0
ds_new["v_lambda"].values[:, -2:, 187, 8] = -2
ds_new["v"].values[:, -2:, 187, 8] = -1.0

In [ ]:
ds_day = ds_new.sel(time="2024-02-29")
ds_day

In [ ]:
ds_day.S.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=30)

In [ ]:
ds_day.S_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=30)

In [ ]:
ds_day.T.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=5)

In [ ]:
ds_day.T_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_day.u.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-0.5, vmax=0.5)

In [ ]:
ds_day.u_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_day.v.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-0.5, vmax=0.5)

In [ ]:
ds_day.v_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_day.S_lambda.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.5), Ny=slice(59.6, 59.8)).where(
    lambda x: x != 0
).plot()

In [ ]:
ds_day.u_lambda.isel(time=-1, Nz=-1).sel(Nx_faces=slice(10.2, 10.5), Ny=slice(59.6, 59.8)).where(
    lambda x: x != 0
).plot()

In [ ]:
ds_day.v_lambda.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.5), Ny_faces=slice(59.6, 59.8)).where(
    lambda x: x != 0
).plot()

In [ ]:
ds_day.S.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.5), Ny=slice(59.6, 59.8)).where(
    lambda x: x != 0
).plot(vmin=-1)

In [ ]:
ds_new.to_netcdf(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_forcing_v2.nc"))